In [1]:
import xarray as xr
import numpy as np
import json
import pandas as pd
import os
import glob

def generate_boxplot_data_meteo():
    print("--- 📊 GENERATOR BACKEND: BOXPLOT STATISTICS (SSRD, TEMP, TP) ---")

    # 1. KONFIGURASI
    start_year = 1979
    end_year = 2024
    data_folder = '.'
    output_filename = 'boxplot_data_meteo.json'
    
    # Definisi pencarian file dan logika konversi unit
    configs = {
        'ssrd': {
            'pat': '*surface_solar_radiation_downwards*', 
            'vars': ['ssrd', 'var169'],
            'unit_conv': lambda x: x / 3600.0 # Joule/m2 -> Watt/m2
        },
        'temp': {
            'pat': '*2m_temperature*', 
            'vars': ['t2m', '2t', 'var167'],
            'unit_conv': lambda x: x - 273.15 # Kelvin -> Celsius
        },
        'tp': {
            'pat': '*total_precipitation*', 
            'vars': ['tp', 'var228'],
            'unit_conv': lambda x: x * 1000.0 # m -> mm
        }
    }

    final_json = {}

    for var_name, cfg in configs.items():
        print(f"\nProcessing Variable: {var_name.upper()}")
        yearly_list = []
        lats, lons = None, None

        for year in range(start_year, end_year + 1):
            try:
                f_list = glob.glob(os.path.join(data_folder, f"{cfg['pat']}*{year}*.nc"))
                if not f_list: continue
                
                ds = xr.open_dataset(f_list[0])
                if 'valid_time' in ds.coords: ds = ds.rename({'valid_time': 'time'})
                
                # Deteksi Nama Variabel
                v_target = next((v for v in cfg['vars'] if v in ds), None)
                if v_target is None: continue
                
                # Load data mentah dan konversi unit
                data_hourly = cfg['unit_conv'](ds[v_target])
                
                if lats is None:
                    lats, lons = data_hourly.latitude.values, data_hourly.longitude.values

                # Hitung 5-number summary (Min, Q1, Median, Q3, Max) per bulan
                # Menggunakan 1MS (Month Start) agar label waktu rapi
                monthly_stats = data_hourly.resample(time='1MS').quantile([0, 0.25, 0.5, 0.75, 1.0], dim='time')
                yearly_list.append(monthly_stats)
                
                print(f"   [{year}] Stats Extracted", end='\r')
                ds.close()
            except Exception as e:
                print(f" Error {year}: {e}")

        if not yearly_list: 
            print(f" Skipping {var_name}: No data found.")
            continue

        # Gabungkan semua tahun dan paksa urutan dimensi: (time, quantile, latitude, longitude)
        print(f"\nAggregating Time Series for {var_name}...")
        ds_all_ts = xr.concat(yearly_list, dim='time').transpose("time", "quantile", "latitude", "longitude")
        
        # Hitung Climatology (12 Bulan) dan Seasonal (4 Musim)
        ds_clim = ds_all_ts.groupby('time.month').mean(dim='time').transpose("month", "quantile", "latitude", "longitude")
        ds_seas = ds_all_ts.groupby('time.season').mean(dim='time').transpose("season", "quantile", "latitude", "longitude")

        def clean_list_2d(arr_2d):
            """Memastikan output array 2D (Waktu, 5) dan bebas NaN/Inf untuk JSON"""
            return [[round(float(v), 3) if (v is not None and np.isfinite(v)) else None for v in row] for row in arr_2d]

        # 1. Hitung Area Average (Representasi Statistik Domain)
        # Sumbu waktu tetap di index pertama setelah operasi mean()
        aa_ts = ds_all_ts.mean(dim=['latitude', 'longitude'], skipna=True).values
        aa_clim = ds_clim.mean(dim=['latitude', 'longitude'], skipna=True).values
        aa_seas = ds_seas.mean(dim=['latitude', 'longitude'], skipna=True).values

        var_output = {
            "metadata": {
                "labels_ts": [pd.to_datetime(t).strftime('%Y-%m') for t in ds_all_ts.time.values],
                "labels_clim": ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"],
                "labels_seas": ["DJF", "MAM", "JJA", "SON"]
            },
            "area_average": {
                "ts": clean_list_2d(aa_ts),
                "clim": clean_list_2d(aa_clim),
                "seas": clean_list_2d(aa_seas)
            },
            "grid_data": {}
        }

        # 2. Isi Data per Grid Point
        print(f"Saving Grid Stats for {var_name}...")
        for i in range(len(lats)):
            for j in range(len(lons)):
                key = f"{i},{j}"
                # Data grid point: (time, 5)
                ts_vals = ds_all_ts.values[:, :, i, j]
                
                # Jika grid ini kosong (land mask / missing), lewatkan
                if np.isnan(ts_vals).all():
                    continue
                
                var_output["grid_data"][key] = {
                    "ts": clean_list_2d(ts_vals),
                    "clim": clean_list_2d(ds_clim.values[:, :, i, j]),
                    "seas": clean_list_2d(ds_seas.values[:, :, i, j])
                }
        
        final_json[var_name] = var_output

    # Simpan Metadata Global
    final_json["metadata"] = {
        "latitudes": lats.tolist() if lats is not None else [],
        "longitudes": lons.tolist() if lons is not None else [],
        "stats_order": ["min", "q1", "median", "q3", "max"]
    }

    with open(output_filename, 'w') as f:
        json.dump(final_json, f)
    
    print(f"\n✅ SELESAI! File '{output_filename}' berhasil dibuat.")

# Jalankan proses
generate_boxplot_data_meteo()

--- 📊 GENERATOR BACKEND: BOXPLOT STATISTICS (SSRD, TEMP, TP) ---

Processing Variable: SSRD
   [2024] Stats Extracted
Aggregating Time Series for ssrd...
Saving Grid Stats for ssrd...

Processing Variable: TEMP
   [2024] Stats Extracted
Aggregating Time Series for temp...
Saving Grid Stats for temp...

Processing Variable: TP
   [2024] Stats Extracted
Aggregating Time Series for tp...
Saving Grid Stats for tp...

✅ SELESAI! File 'boxplot_data_meteo.json' berhasil dibuat.
